# inaugural_convergence_v1

Fresh LSTM and FFNN fits; validation selects the restored best checkpoint. A patience stop is a validation plateau; a run reaching the epoch ceiling is budget-limited, not proven converged.

Run cells in order on a T4 GPU. Authorize Google Drive directly in the browser. The original pilot and imports remain unchanged. Prepared manifests contain the derived quarantine audit.

The bundle checksum and training settings below are frozen together. Do not alter them to resume a different experiment. Cross-budget vocabularies change prediction support.


In [ ]:
from pathlib import Path

EPOCHS = 20
PATIENCE = 3
BUDGETS = [10, 50, 100]
SEEDS = [42, 43, 44]
CORPORA = ['inaugural']
USE_DRIVE = True
REQUIRE_GPU = True
OUTPUT_NAME = 'inaugural_convergence_v1'
WORKSPACE = Path("/content/nlp_assignment")
EXPECTED_BUNDLE_SHA256 = 'bfe0ec10ee45a06c5c3f428e395abee89b3a15413013160a9a58622cfe77fcfc'

assert EPOCHS > 0 and PATIENCE > 0
assert CORPORA and SEEDS and BUDGETS
assert Path(OUTPUT_NAME).name == OUTPUT_NAME
print("Experiment:", CORPORA, "epochs:", EPOCHS, "seeds:", SEEDS)


In [ ]:
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile

import torch
from google.colab import files

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun the notebook.")
print("Python:", sys.version)
print("Torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one matching prepared bundle ZIP.")
BUNDLE_PATH = Path(next(iter(uploaded))).resolve()
if BUNDLE_PATH.suffix.lower() != ".zip":
    raise ValueError("Expected the prepared ZIP bundle, not a notebook or individual corpus file.")
bundle_digest = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
if bundle_digest != EXPECTED_BUNDLE_SHA256:
    raise ValueError("Input bundle checksum differs from this notebook; rebuild a matching notebook/bundle pair.")
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for member in archive.infolist():
        target = (WORKSPACE / member.filename).resolve()
        if not target.is_relative_to(WORKSPACE.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("Unsafe archive entry: " + member.filename)
    archive.extractall(WORKSPACE)
bundle_manifest = json.loads((WORKSPACE / "bundle_manifest.json").read_text())
for name, expected in bundle_manifest["files"].items():
    target = (WORKSPACE / name).resolve()
    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("Unsafe bundle manifest entry")
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError("Bundle file checksum mismatch: " + name)
if set(bundle_manifest["corpora"]) != set(CORPORA):
    raise ValueError("Bundle corpora differ from the selected experiment.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6.0", "pandas>=2.2", "matplotlib>=3.9", "datasketch>=1.6", "nltk>=3.9"])
os.chdir(WORKSPACE)
print("Verified bundle:", bundle_digest)
print("Prepared corpora:", bundle_manifest["corpora"])
print("Workspace:", WORKSPACE)

In [ ]:
import yaml
from google.colab import drive

if USE_DRIVE:
    drive.mount("/content/drive")
    OUTPUT = Path("/content/drive/MyDrive/NLP_Experiments") / OUTPUT_NAME
else:
    OUTPUT = Path("/content/results") / OUTPUT_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load((WORKSPACE / "input/suite.yaml").read_text())
if set(config["corpora"]) != set(CORPORA):
    raise ValueError("The bundled configuration must contain exactly the selected corpora.")
if config["training"]["epochs"] != EPOCHS or config["training"]["patience"] != PATIENCE:
    raise ValueError("Stopping settings differ from the frozen bundle. Build a new matching bundle/notebook.")
if config["budgets_percent"] != BUDGETS:
    raise ValueError("Budgets differ from the frozen bundle.")
config["output_root"] = str(OUTPUT)
for specification in config["models"]:
    if specification["family"] == "neural":
        if specification["final_seeds"] != list(SEEDS):
            raise ValueError("Seeds differ from the frozen bundle.")
EFFECTIVE_CONFIG = WORKSPACE / "input/effective_suite.yaml"
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
retained_bundle = OUTPUT / "input_bundle.zip"
if retained_bundle.exists():
    if hashlib.sha256(retained_bundle.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise ValueError("A different input bundle already exists. Choose a new OUTPUT_NAME.")
else:
    shutil.copyfile(BUNDLE_PATH, retained_bundle)
shutil.copyfile(WORKSPACE / "bundle_manifest.json", OUTPUT / "bundle_manifest.json")
shutil.copyfile(EFFECTIVE_CONFIG, OUTPUT / "effective_suite.yaml")
if (WORKSPACE / "input/audits").exists():
    shutil.copytree(WORKSPACE / "input/audits", OUTPUT / "audits", dirs_exist_ok=True)
print("Durable output:", OUTPUT)
print("Budgets:", config["budgets_percent"])
print("Models:", [specification["id"] for specification in config["models"]])
print("Original test bytes are preserved. Inspect prepared manifests for the experiment's repair audit.")

In [ ]:
stages = ["train", "seal", "evaluate", "report"]
with (OUTPUT / "colab_execution.log").open("a", encoding="utf-8") as execution_log:
    for stage in stages:
        command = [sys.executable, "-u", "src/run.py", "--suite", str(EFFECTIVE_CONFIG), "--stage", stage, "--resume"]
        heading = "\nSTAGE: " + stage + "\n"
        print(heading, flush=True)
        execution_log.write(heading)
        execution_log.flush()
        process = subprocess.Popen(command, cwd=WORKSPACE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            execution_log.write(line)
            execution_log.flush()
        exit_code = process.wait()
        if exit_code != 0:
            subprocess.run([sys.executable, "-u", "src/run.py", "--suite", str(EFFECTIVE_CONFIG), "--stage", "report"], cwd=WORKSPACE, check=False)
            raise RuntimeError("Stage " + stage + " failed; saved artifacts and logs are in " + str(OUTPUT))
print("Experiment complete. Reports, model checkpoints, vocabularies, histories, audits and logs:", OUTPUT)

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

summary_path = OUTPUT / "summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("No completed results yet. Inspect colab_execution.log and per-run attempt logs.")
for corpus in CORPORA:
    display(Markdown("## " + corpus))
    for filename in ("perplexity.png", "vocabulary.png", "oov.png"):
        image_path = OUTPUT / corpus / "figures" / filename
        if image_path.exists():
            display(Image(filename=str(image_path)))
    histories = sorted((OUTPUT / corpus).glob("budget_*/final/lstm/seed*/training.png"))
    if histories:
        display(Image(filename=str(histories[-1])))
print("Complete offline report:", OUTPUT / "report.html")
print("All runs:", OUTPUT / "runs.csv")
print("Seed summaries:", OUTPUT / "summary.csv")
print("Frequency/novelty diagnostics:", OUTPUT / "diagnostics.csv")
print("Reproducible cases:", OUTPUT / "examples.json")

In [ ]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
artifact_hashes = {}
for artifact_path in sorted(OUTPUT.rglob("*")):
    if artifact_path.is_file() and artifact_path.name != "artifact_manifest.json":
        digest = hashlib.sha256()
        with artifact_path.open("rb") as artifact_stream:
            for block in iter(lambda: artifact_stream.read(1024 * 1024), b""):
                digest.update(block)
        artifact_hashes[artifact_path.relative_to(OUTPUT).as_posix()] = digest.hexdigest()
(OUTPUT / "artifact_manifest.json").write_text(json.dumps({"generated_utc": timestamp, "files": artifact_hashes}, indent=2, sort_keys=True) + "\n")
archive_base = Path("/content") / (OUTPUT_NAME + "_results_" + timestamp)
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT.parent, base_dir=OUTPUT.name))
if USE_DRIVE:
    retained_archive = OUTPUT.parent / archive_path.name
    shutil.copyfile(archive_path, retained_archive)
    print("Full result archive retained on Drive:", retained_archive)
print("Full result archive:", archive_path)
print("Includes vocabularies, token counts, manifests, histories, checkpoints, logs, audits, graphs, metrics, diagnostics and examples.")
files.download(str(archive_path))